In [1]:
# =========================
# 0_FNS
# =========================
import os
import shlex
import subprocess
from pathlib import Path
from tqdm import tqdm
import pandas as pd


# -----######-----######-----######-----######-----######-----######-----######
# CORE IMPORTABLE FUNCTION
# _vid_2201_i1_GET_df_mp4_compress
# -----######-----######-----######-----######-----######-----######-----######


def _vid_2201_i1_GET_df_mp4_compress(
    x,
    out_dir="",
    pct_reduce=30,
    mode="bitrate",     # "bitrate" (target ~pct smaller) or "crf" (best visual)
    crf=20,             # used if mode="crf" (18-22 sweet spot)
    preset="slow",
    keep_audio="y",     # "y" copies audio, "n" re-encodes audio to aac 128k
    overwrite="n",
    df_col_path="Path",
    df_col_out="Path_mp4_light",
    add_stats_cols="y",
    verbose="n",
):
    """
    Compress mp4(s) so they are less heavy with minimal visible loss.

    Inputs:
      - x can be:
          1) a single mp4 path (str/Path)
          2) list/tuple of mp4 paths
          3) a pandas DataFrame with a column holding mp4 paths (df_col_path)

    Modes:
      - mode="crf": best perceptual quality compression (not a fixed %)
      - mode="bitrate": targets ~pct_reduce smaller by reducing VIDEO bitrate
                        (audio copied by default, so final reduction depends on audio share)

    Returns:
      - If x is DataFrame: returns df with output path column appended
      - Else: returns list of output paths
    """

    def _run(cmd_list):
        if verbose.lower().startswith("y"):
            print("CMD:", " ".join(shlex.quote(c) for c in cmd_list))
        p = subprocess.run(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return p.returncode, p.stdout, p.stderr

    def _need_ffmpeg():
        for bin_name in ("ffmpeg", "ffprobe"):
            rc, _, _ = _run([bin_name, "-version"])
            if rc != 0:
                raise RuntimeError(
                    f"Missing '{bin_name}'. Install ffmpeg (includes ffprobe).\n"
                    f"Mac (Homebrew): brew install ffmpeg"
                )

    def _ffprobe_json(path):
        cmd = [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration,bit_rate",
            "-of", "default=noprint_wrappers=1:nokey=0",
            str(path),
        ]
        rc, out, err = _run(cmd)
        if rc != 0:
            raise RuntimeError(f"ffprobe failed for: {path}\n{err}")
        # parse simple key=val lines
        info = {}
        for line in out.splitlines():
            if "=" in line:
                k, v = line.split("=", 1)
                info[k.strip()] = v.strip()
        return info

    def _safe_out_path(in_path, out_dir_):
        in_path = Path(in_path)
        if out_dir_:
            out_root = Path(os.path.expanduser(str(out_dir_))).resolve()
        else:
            out_root = in_path.parent.resolve()

        out_root.mkdir(parents=True, exist_ok=True)
        out_name = f"{in_path.stem}__light.mp4"
        return (out_root / out_name).resolve()

    def _compress_one(in_path, out_path):
        in_path = Path(in_path).resolve()
        if not in_path.exists():
            return {"ok": False, "in_path": str(in_path), "out_path": str(out_path), "error": "file_not_found"}

        if out_path.exists() and not overwrite.lower().startswith("y"):
            return {"ok": True, "in_path": str(in_path), "out_path": str(out_path), "error": ""}

        # audio settings
        if keep_audio.lower().startswith("y"):
            aud = ["-c:a", "copy"]
        else:
            aud = ["-c:a", "aac", "-b:a", "128k"]

        # core encode choices
        if str(mode).lower() == "crf":
            # perceptual approach: smallest file for a given look (recommended)
            cmd = [
                "ffmpeg",
                "-hide_banner",
                "-y" if overwrite.lower().startswith("y") else "-n",
                "-i", str(in_path),
                "-c:v", "libx264",
                "-preset", str(preset),
                "-crf", str(int(crf)),
                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                *aud,
                str(out_path),
            ]
            rc, _, err = _run(cmd)
            if rc != 0:
                return {"ok": False, "in_path": str(in_path), "out_path": str(out_path), "error": err.strip()}

        else:
            # bitrate targeting: aim ~pct_reduce smaller video bitrate
            info = _ffprobe_json(in_path)
            br = info.get("bit_rate", "")
            if not br or not br.isdigit():
                return {"ok": False, "in_path": str(in_path), "out_path": str(out_path), "error": "missing_bitrate"}

            br = int(br)  # total bitrate (bits/sec) container-level
            # We'll reduce VIDEO bitrate by pct_reduce. Since audio is often copied,
            # final file reduction will be close-ish but not exact.
            factor = max(0.05, 1.0 - (float(pct_reduce) / 100.0))
            target_v_bps = int(br * factor)

            # clamp to something sane
            target_v_bps = max(200_000, target_v_bps)

            cmd = [
                "ffmpeg",
                "-hide_banner",
                "-y" if overwrite.lower().startswith("y") else "-n",
                "-i", str(in_path),
                "-c:v", "libx264",
                "-preset", str(preset),
                "-b:v", str(target_v_bps),
                "-maxrate", str(int(target_v_bps * 1.15)),
                "-bufsize", str(int(target_v_bps * 2)),
                "-pix_fmt", "yuv420p",
                "-movflags", "+faststart",
                *aud,
                str(out_path),
            ]
            rc, _, err = _run(cmd)
            if rc != 0:
                return {"ok": False, "in_path": str(in_path), "out_path": str(out_path), "error": err.strip()}

        # stats
        out_size = out_path.stat().st_size if out_path.exists() else 0
        in_size = in_path.stat().st_size if in_path.exists() else 0
        ratio = (out_size / in_size) if in_size else None

        return {
            "ok": True,
            "in_path": str(in_path),
            "out_path": str(out_path),
            "in_mb": round(in_size / (1024**2), 3) if in_size else None,
            "out_mb": round(out_size / (1024**2), 3) if out_size else None,
            "ratio_out_in": round(ratio, 4) if ratio is not None else None,
            "error": "",
        }

    # --- start ---
    _need_ffmpeg()

    # normalize input
    is_df = isinstance(x, pd.DataFrame)
    if is_df:
        df = x.copy()
        paths = df[df_col_path].tolist()
    elif isinstance(x, (list, tuple)):
        paths = list(x)
        df = None
    else:
        paths = [x]
        df = None

    results = []
    for p in tqdm(paths, desc="mp4 compress", total=len(paths)):
        in_path = Path(os.path.expanduser(str(p))).resolve()
        out_path = _safe_out_path(in_path, out_dir)
        res = _compress_one(in_path, out_path)
        results.append(res)

    # return format
    out_paths = [r.get("out_path", "") for r in results]

    if is_df:
        df[df_col_out] = out_paths
        if add_stats_cols.lower().startswith("y"):
            df["mp4_in_mb"] = [r.get("in_mb") for r in results]
            df["mp4_out_mb"] = [r.get("out_mb") for r in results]
            df["mp4_ratio_out_in"] = [r.get("ratio_out_in") for r in results]
            df["mp4_ok"] = [r.get("ok") for r in results]
            df["mp4_error"] = [r.get("error") for r in results]
        return df

    return out_paths


In [5]:
out_paths = _vid_2201_i1_GET_df_mp4_compress(
    "/Users/yerik/Desktop/bnb/7cfd6335dd5d442eae9f0e24a98ebe2b.MP4",
    out_dir="/Users/yerik/Desktop/bnb/",   # "" => same folder
    mode="crf",
    crf=40,
    preset="slow",
    keep_audio="y",
    overwrite="n",
)

mp4 compress: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:50<00:00, 50.52s/it]


In [3]:
out_paths = _vid_2201_i1_GET_df_mp4_compress(
    "/Users/yerik/Desktop/NEWmagicHOUSE_animations/__promo_vid_mockUP/_this one .mp4",
    out_dir="/Users/yerik/Desktop/NEWmagicHOUSE_animations/__promo_vid_mockUP/", 
    mode="bitrate",
    pct_reduce=30,
    preset="slow",
    keep_audio="y",
    overwrite="n",
)

mp4 compress: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:25<00:00, 25.44s/it]


# square it 

In [6]:
import subprocess
from pathlib import Path

# INPUT
in_path = Path("/Users/yerik/Desktop/NEWmagicHOUSE_animations/__promo_vid_mockUP/_this one __light.mp4")
out_path = in_path.with_name(in_path.stem + "_SQUARE.mp4")

# ffmpeg crop:
# - square = min(width, height)
# - center crop (equal left/right)
cmd = [
    "ffmpeg",
    "-y",
    "-i", str(in_path),
    "-vf", "crop=min(iw\\,ih):min(iw\\,ih):(iw-min(iw\\,ih))/2:(ih-min(iw\\,ih))/2",
    "-c:a", "copy",
    str(out_path)
]

subprocess.run(cmd, check=True)

print(f"✅ Square video created:\n{out_path}")


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.0.13.3)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/7.1.1_3 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags='-Wl,-ld_classic' --enable-ffplay --enable-gnutls --enable-gpl --enable-libaom --enable-libaribb24 --enable-libbluray --enable-libdav1d --enable-libharfbuzz --enable-libjxl --enable-libmp3lame --enable-libopus --enable-librav1e --enable-librist --enable-librubberband --enable-libsnappy --enable-libsrt --enable-libssh --enable-libsvtav1 --enable-libtesseract --enable-libtheora --enable-libvidstab --enable-libvmaf --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-lzma --enable-libfontconfig --enable-libfreetype --enable-frei0r --enable-libass --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-libspeex

✅ Square video created:
/Users/yerik/Desktop/NEWmagicHOUSE_animations/__promo_vid_mockUP/_this one __light_SQUARE.mp4


[out#0/mp4 @ 0xc9100c840] video:12140KiB audio:646KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.398226%
frame= 1386 fps=184 q=-1.0 Lsize=   12837KiB time=00:00:46.17 bitrate=2277.2kbits/s speed=6.13x    
[libx264 @ 0xc90c89180] frame I:10    Avg QP:19.66  size:141298
[libx264 @ 0xc90c89180] frame P:411   Avg QP:21.76  size: 19409
[libx264 @ 0xc90c89180] frame B:965   Avg QP:27.87  size:  3151
[libx264 @ 0xc90c89180] consecutive B-frames:  2.7% 12.1%  3.5% 81.7%
[libx264 @ 0xc90c89180] mb I  I16..4:  6.3% 39.2% 54.5%
[libx264 @ 0xc90c89180] mb P  I16..4:  2.1%  4.6%  1.5%  P16..4: 34.9% 13.1%  8.0%  0.0%  0.0%    skip:35.9%
[libx264 @ 0xc90c89180] mb B  I16..4:  0.2%  0.4%  0.1%  B16..8: 24.4%  2.1%  0.4%  direct: 1.4%  skip:70.9%  L0:40.2% L1:54.1% BI: 5.6%
[libx264 @ 0xc90c89180] 8x8 transform intra:52.8% inter:67.0%
[libx264 @ 0xc90c89180] coded y,uvDC,uvAC intra: 54.9% 67.7% 34.8% inter: 8.9% 14.1% 0.7%
[libx264 @ 0xc90c89180] i16 v,h,dc,p: 30% 35% 11% 